# 03. Budget-Constrained Optimization & Knapsack Formulation
**Team Member**: Optimization Analyst (Member 3)
**Objective**: Formulate the 0-1 Knapsack problem using ITE estimates, solve via SciPy MILP, and benchmark against heuristic targeting policies.

In [ ]:
import pandas as pd
import numpy as np
from src.data.synthetic import generate_synthetic_data
from src.models.dml_estimator import DoubleMLEstimator
from src.optimization.knapsack import solve_knapsack_discount_allocation
from src.optimization.policy_allocator import compare_allocation_policies

# Generate data and obtain ITE estimates
df = generate_synthetic_data(n_samples=5000, random_state=42)
features = ["Age", "PriorSpend", "VisitFrequency", "LoyaltyScore"]
dml = DoubleMLEstimator(n_splits=3, nuisance_type="random_forest", backend="native").fit(df[features], df["Discount"], df["Sales"])
df["Estimated_ITE"] = dml.ite_estimates_
print("ITE estimation ready for optimization.")

In [ ]:
# Solve 0-1 Knapsack with a $5,000 budget
knapsack_res = solve_knapsack_discount_allocation(
    ite_estimates=df["Estimated_ITE"].values,
    item_prices=df["EstimatedItemPrice"].values,
    budget=5000.0,
    discount_rate=0.15,
    method="milp"
)
print(f"Targeted Customers: {knapsack_res['n_targeted']}")
print(f"Total Cost: ${knapsack_res['total_cost']:.2f}")
print(f"Expected Incremental Gain: ${knapsack_res['total_incremental_value']:.2f}")
print(f"ROI: {knapsack_res['estimated_roi']:.2f}x")

In [ ]:
# Compare Causal Optimal vs Heuristic vs Random
policy_comparison = compare_allocation_policies(df, df["Estimated_ITE"].values, budget=5000.0)
policy_comparison